In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CBEPC10", con=connection2)

connection2.close()




In [2]:
base2

,estprogcitcod,estprogcitnom
0,1,EN PROGRAMACION
1,2,APROBADA
2,4,SUSPENDIDA
3,3,BLOQUEADA


In [3]:
a=base2[base2.duplicated(subset=["estprogcitcod"])]
a

,estprogcitcod,estprogcitnom


In [4]:
base2.columns

Index(['estprogcitcod', 'estprogcitnom'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cbepc10 ()')
base2.to_sql(name='tmp_cbepc10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbepc10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cbepc10 
ALTER COLUMN	estprogcitcod	TYPE	CHARACTER (1),
ALTER COLUMN	estprogcitnom	TYPE	CHARACTER (20);


UPDATE cbepc10 
SET 

estprogcitnom                 = t.estprogcitnom


FROM tmp_cbepc10 t 
WHERE cbepc10.estprogcitcod = t.estprogcitcod and cbepc10.estprogcitcod IS NOT NULL and t.estprogcitcod IS NOT NULL ;

INSERT INTO cbepc10 
(estprogcitcod, estprogcitnom) 
SELECT 
estprogcitcod, estprogcitnom

FROM tmp_cbepc10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbepc10 
    WHERE cbepc10.estprogcitcod = tmp_cbepc10.estprogcitcod and cbepc10.estprogcitcod IS NOT NULL and tmp_cbepc10.estprogcitcod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbepc10;
DELETE FROM cbepc10 WHERE estprogcitcod ='';
"""


c2= text(query2)
connection3.execute(c2)
base2 = pd.read_sql_query(f"SELECT * FROM cbepc10", con=connection3)

connection3.close()


In [6]:
base2.columns

Index(['estprogcitcod', 'estprogcitnom'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cbepc10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""

ALTER TABLE tmp_cbepc10 
ALTER COLUMN	estprogcitcod	TYPE	CHARACTER (1),
ALTER COLUMN	estprogcitnom	TYPE	CHARACTER (20);


UPDATE dim_estpro 
SET 

des_est       = t.estprogcitnom


FROM tmp_cbepc10 t 
WHERE dim_estpro.cod_est = t.estprogcitcod AND dim_estpro.cod_est  IS NOT NULL AND t.estprogcitcod  IS NOT NULL;


INSERT INTO dim_estpro (cod_est, des_est) 
SELECT estprogcitcod, estprogcitnom      

FROM tmp_cbepc10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_estpro 
    WHERE dim_estpro.cod_est = tmp_cbepc10.estprogcitcod
);

DROP TABLE tmp_cbepc10;

"""



c1= text(query)
connection4.execute(c1)
connection4.close()